# GPS Flight Path Analysis
Load GPS tracker data and export a KML file for viewing the 3D flight path in Google Earth.

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

EXPORTS_DIR = Path(__file__).resolve().parents[1] / "exports" if "__file__" in dir() else Path.cwd().parent / "exports"

# Load GPS data
gps_path = sorted(EXPORTS_DIR.glob("GPSTrk*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)[0]
gps = pd.read_csv(gps_path)
gps.columns = gps.columns.str.strip()

# Convert key columns to numeric
for col in ["ALT", "LAT", "LON", "Altitude AGL", "HORZV", "VERTV"]:
    gps[col] = pd.to_numeric(gps[col], errors="coerce")

# Detect launch/apogee/landing rows from boolean flag columns
gps["Launch detection"] = gps["Launch detection"].astype(str).str.strip().str.upper() == "TRUE"
gps["Apogee detection"] = gps["Apogee detection"].astype(str).str.strip().str.upper() == "TRUE"
gps["Landing detection"] = gps["Landing detection"].astype(str).str.strip().str.upper() == "TRUE"

apogee_idx = gps.index[gps["Apogee detection"]].min() if gps["Apogee detection"].any() else gps["ALT"].idxmax()

print(f"GPS file: {gps_path.name}")
print(f"Rows: {len(gps)}")
print(f"Lat range: {gps['LAT'].min():.5f} – {gps['LAT'].max():.5f}")
print(f"Lon range: {gps['LON'].min():.5f} – {gps['LON'].max():.5f}")
print(f"Alt range: {gps['ALT'].min():.1f} – {gps['ALT'].max():.1f} m")
print(f"Apogee row: {apogee_idx}")
gps.head()

GPS file: GPSTrk05406_03-21-2026_11_52_18.csv
Rows: 597
Lat range: 35.34264 – 35.35117
Lon range: -117.83723 – -117.80932
Alt range: 1980.9 – 26594.1 m
Apogee row: 301


,UTCTIME,UNIXTIME,ALT,LAT,LON,#SATS,FIX,HORZV,VERTV,HEAD,...,>40,>32,>24,RSSI,BATT,Altitude AGL,Launch detection,Apogee detection,Landing detection,Distance (feet)
0,Mar 21 2026 18:52:12.899 UTC,1.774141e+09,2053.9,35.34768,-117.80932,32,3,0,0,135,...,4,12,6,-41,3.913,0.9,False,False,False,0.0
1,Mar 21 2026 18:52:13.000 UTC,1.774141e+09,2053.9,35.34768,-117.80932,32,3,0,0,4,...,4,13,5,-41,3.891,0.9,False,False,False,0.0
2,Mar 21 2026 18:52:13.199 UTC,1.774141e+09,2053.9,35.34768,-117.80932,32,3,0,0,-53,...,4,13,5,-43,3.891,0.9,False,False,False,0.0
3,Mar 21 2026 18:52:13.299 UTC,1.774141e+09,2053.9,35.34768,-117.80932,32,3,0,0,36,...,4,13,5,-43,3.891,0.9,False,False,False,0.0
4,Mar 21 2026 18:52:13.399 UTC,1.774141e+09,2053.9,35.34768,-117.80932,32,3,0,0,107,...,4,13,5,-43,3.891,0.9,False,False,False,0.0


In [14]:
import xml.etree.ElementTree as ET

def build_kml(df: pd.DataFrame, name: str = "SnowbirdPrim GPS Flight Path") -> str:
    """Build a KML string with the flight path as a 3-D LineString and key waypoints."""
    kml = ET.Element("kml", xmlns="http://www.opengis.net/kml/2.2")
    doc = ET.SubElement(kml, "Document")
    ET.SubElement(doc, "name").text = name

    # --- Line style ---
    style = ET.SubElement(doc, "Style", id="flightPath")
    ls = ET.SubElement(style, "LineStyle")
    ET.SubElement(ls, "color").text = "ff0000ff"  # red (AABBGGRR)
    ET.SubElement(ls, "width").text = "3"

    # --- Flight path line ---
    pm = ET.SubElement(doc, "Placemark")
    ET.SubElement(pm, "name").text = "Flight Path"
    ET.SubElement(pm, "styleUrl").text = "#flightPath"
    line = ET.SubElement(pm, "LineString")
    ET.SubElement(line, "extrude").text = "1"
    ET.SubElement(line, "tessellate").text = "1"
    ET.SubElement(line, "altitudeMode").text = "absolute"
    coords_text = "\n".join(
        f"{row.LON},{row.LAT},{row.ALT}" for row in df.itertuples()
    )
    ET.SubElement(line, "coordinates").text = coords_text

    # --- Waypoint helper ---
    def add_point(name_text, lat, lon, alt):
        wp = ET.SubElement(doc, "Placemark")
        ET.SubElement(wp, "name").text = name_text
        pt = ET.SubElement(wp, "Point")
        ET.SubElement(pt, "altitudeMode").text = "absolute"
        ET.SubElement(pt, "coordinates").text = f"{lon},{lat},{alt}"

    # Launch point
    first = df.iloc[0]
    add_point("Launch", first.LAT, first.LON, first.ALT)

    # Apogee
    apo = df.loc[df["ALT"].idxmax()]
    add_point(f"Apogee ({apo.ALT:.0f} m)", apo.LAT, apo.LON, apo.ALT)

    # Landing
    last = df.iloc[-1]
    add_point("Landing", last.LAT, last.LON, last.ALT)

    return '<?xml version="1.0" encoding="UTF-8"?>\n' + ET.tostring(kml, encoding="unicode")

kml_string = build_kml(gps)

# Write KML to exports/
kml_path = EXPORTS_DIR / f"{gps_path.stem}_flight_path.kml"
kml_path.write_text(kml_string, encoding="utf-8")
print(f"KML saved to: {kml_path}")
print("Open this file in Google Earth to see the 3-D flight path.")

KML saved to: c:\Users\lamor\OneDrive\src\DataAnalysisSnowbirdTest\exports\GPSTrk05406_03-21-2026_11_52_18_flight_path.kml
Open this file in Google Earth to see the 3-D flight path.


In [15]:
import plotly.graph_objects as go

# Color by altitude for visual depth
fig = go.Figure()

# Flight path line
fig.add_trace(go.Scattermap(
    lat=gps["LAT"],
    lon=gps["LON"],
    mode="lines",
    line=dict(width=3, color="red"),
    name="Flight Path",
    text=[f"Alt: {a:.0f} m" for a in gps["ALT"]],
    hoverinfo="text+name",
))

# Key waypoints
launch = gps.iloc[0]
apo = gps.loc[gps["ALT"].idxmax()]
landing = gps.iloc[-1]

fig.add_trace(go.Scattermap(
    lat=[launch.LAT], lon=[launch.LON], mode="markers+text",
    marker=dict(size=12, color="green"), name="Launch",
    text=["Launch"], textposition="top right",
))
fig.add_trace(go.Scattermap(
    lat=[apo.LAT], lon=[apo.LON], mode="markers+text",
    marker=dict(size=12, color="blue"), name=f"Apogee ({apo.ALT:.0f} m)",
    text=[f"Apogee\n{apo.ALT:.0f} m"], textposition="top right",
))
fig.add_trace(go.Scattermap(
    lat=[landing.LAT], lon=[landing.LON], mode="markers+text",
    marker=dict(size=12, color="orange"), name="Landing",
    text=["Landing"], textposition="top right",
))

fig.update_layout(
    map=dict(
        style="open-street-map",
        center=dict(lat=gps["LAT"].mean(), lon=gps["LON"].mean()),
        zoom=13,
    ),
    title="SnowbirdPrim GPS Flight Path (2D Map)",
    height=600,
    margin=dict(l=0, r=0, t=40, b=0),
    showlegend=True,
)
fig.show()

In [16]:
# 3D flight path — Lat/Lon/Alt scatter with altitude coloring
fig3d = go.Figure()

fig3d.add_trace(go.Scatter3d(
    x=gps["LON"], y=gps["LAT"], z=gps["ALT"],
    mode="lines",
    line=dict(width=4, color=gps["ALT"], colorscale="Hot", showscale=True,
              colorbar=dict(title="Alt (m)")),
    name="Flight Path",
    hovertext=[f"Alt: {a:.0f} m<br>Lat: {la:.5f}<br>Lon: {lo:.5f}"
               for a, la, lo in zip(gps["ALT"], gps["LAT"], gps["LON"])],
    hoverinfo="text",
))

# Waypoint markers
for label, row, color in [("Launch", launch, "green"), (f"Apogee ({apo.ALT:.0f} m)", apo, "blue"), ("Landing", landing, "orange")]:
    fig3d.add_trace(go.Scatter3d(
        x=[row.LON], y=[row.LAT], z=[row.ALT],
        mode="markers+text", text=[label], textposition="top center",
        marker=dict(size=6, color=color),
        name=label,
    ))

fig3d.update_layout(
    scene=dict(
        xaxis_title="Longitude",
        yaxis_title="Latitude",
        zaxis_title="Altitude (m)",
        aspectmode="data",
    ),
    title="SnowbirdPrim GPS Flight Path (3D)",
    height=700,
    margin=dict(l=0, r=0, t=40, b=0),
)
fig3d.show()